# Dimensionamiento de un sistema de colas

## Tipo de problema

Dimensionar el número de cajas registradoras o servidores de atención telefónica para minimizar el tiempo de espera del cliente sin disparar los costes de personal.

## Formas de resolverlo

Los nombres de la primera columna coinciden con la [tabla del README](../README.md#formas-de-resolverlo).

| Enfoque (README) | Aplicación en este problema | Uso en este ejercicio |
|---|---|---|
| Fórmulas analíticas | Calcular métricas de una cola M/M/c con Erlang C en régimen estable. | **Principal** |
| Modelos estocásticos y simulación | Simular eventos discretos cuando horarios, abandonos o distribuciones complican el sistema. | Alternativa |
| Enumeración | Comparar el coste de cada dotación de uno a seis empleados. | **Complementario** |

**Enfoque utilizado:** Fórmulas analíticas.

**Enfoque complementario:** Enumeración.

**Técnica concreta:** Erlang C para cada dotación estable y comparación de costes entre uno y seis empleados.

## Ejercicio propuesto

Dimensionar un servicio con llegadas de 8 clientes/h y atención de 5/h  
por empleado. Entre 1 y 6 empleados. Coste de personal: 12 €/empleado/h;  
se valora cada hora de espera de un cliente en 15 €.  
Objetivo: minimizar 12*c + 15*Lq, donde Lq es el número medio en cola.  
Método: fórmula Erlang C de una cola M/M/c y enumeración de c.  
Supuestos: llegadas Poisson, servicios exponenciales independientes,  
una cola FIFO, sin abandonos, capacidad de espera infinita, régimen estable.  
Los sistemas con lambda >= c*mu no tienen equilibrio y se descartan.

## Método elegido para este ejercicio

**Enfoque utilizado:** Fórmulas analíticas.

**Enfoque complementario:** Enumeración.

Se calculan las métricas M/M/c mediante Erlang C y se comparan entre uno y seis servidores. Se minimiza personal más coste de espera. Se descartan sistemas inestables, cuya tasa de llegadas alcanza o supera la capacidad de servicio.

## Implementación

Ejecuta las celdas de código en orden. Solo se utiliza la biblioteca estándar de Python.

In [1]:
from math import factorial


def metricas(llegadas, servicio, servidores):
    if llegadas < 0 or servicio <= 0 or servidores < 1:
        raise ValueError('Se requieren lambda >= 0, mu > 0 y c >= 1')
    rho = llegadas / (servidores * servicio)
    if rho >= 1:
        return None
    carga = llegadas / servicio
    cola = carga ** servidores / (factorial(servidores) * (1 - rho))
    p0 = 1 / (sum(carga ** k / factorial(k) for k in range(servidores)) + cola)
    prob_espera = cola * p0
    lq = prob_espera * rho / (1 - rho)
    wq = lq / llegadas if llegadas else 0.0
    return rho, prob_espera, lq, wq


def resolver():
    opciones = []
    for c in range(1, 7):
        resultado = metricas(8, 5, c)
        if resultado is not None:
            rho, probabilidad, lq, wq = resultado
            opciones.append((12 * c + 15 * lq, c, rho, probabilidad, wq))
    return min(opciones), opciones

## Resolución y resultados

In [2]:
mejor, opciones = resolver()
for coste, c, rho, probabilidad, wq in opciones:
    print(f'{c} empleados: coste={coste:.2f} €/h; utilización={rho:.1%}; '
          f'P(esperar)={probabilidad:.1%}; espera media={60 * wq:.2f} min')
print(f'Óptimo entre 1 y 6 empleados: {mejor[1]}, coste {mejor[0]:.2f} €/h')

2 empleados: coste=66.67 €/h; utilización=80.0%; P(esperar)=71.1%; espera media=21.33 min
3 empleados: coste=40.69 €/h; utilización=53.3%; P(esperar)=27.4%; espera media=2.35 min
4 empleados: coste=48.91 €/h; utilización=40.0%; P(esperar)=9.1%; espera media=0.45 min
5 empleados: coste=60.18 €/h; utilización=32.0%; P(esperar)=2.6%; espera media=0.09 min
6 empleados: coste=72.03 €/h; utilización=26.7%; P(esperar)=0.6%; espera media=0.02 min
Óptimo entre 1 y 6 empleados: 3, coste 40.69 €/h


## Interpretación

Elegir **3 empleados**, con coste de **40,69 €/h** y espera media de **2,35 min** por cliente. Se presupone régimen estable, llegadas Poisson, servicios exponenciales y una cola sin abandonos. Es óptimo entre las seis dotaciones consideradas.

**Alcance:** los datos son sintéticos. Las garantías y supuestos se refieren a esta instancia; al aumentar su tamaño, la enumeración puede resultar costosa.